In [ ]:
# from datasets import load_dataset

# # Wikipedia RU
# wiki = load_dataset("wikimedia/wikipedia", "20231101.ru", split="train")
# # Save to folder
# wiki.save_to_disk("d:/Neuro_net/Projects/LLM/Gen/data/wiki_ru_20231101")

# OSCAR RU
# oscar_russian = load_dataset("oscar-corpus/oscar", "ru", split="train")

# oscar_russian

##### Download Wiki

In [1]:
from datasets import load_from_disk
wiki = load_from_disk("raw_datasets/wiki_ru_20231101")
wiki_ds = wiki.remove_columns([c for c in wiki.column_names if c != "text"])  # .to_pandas()
wiki_ds

Loading dataset from disk:   0%|          | 0/21 [00:00<?, ?it/s]

Dataset({
    features: ['text'],
    num_rows: 1945063
})

In [ ]:
import re 
import hashlib

CLEAN_RE = re.compile(r'[^a-zа-яё0-9 ]')
SOURCE_RE = re.compile(r"//[^//]+//")
seen = set()

def repetition_score(text):
    tokens = CLEAN_RE.sub(' ', text.lower()).split()
    return len(set(tokens)) / len(tokens) if tokens else 0.0


def is_good_example(example, min_len=200, threshold=0.6):
    text = example["text"]

    if not isinstance(text, str) or len(text) <= min_len:
        return False
    if "{" in text or "}" in text:
        return False
    if SOURCE_RE.search(text):
        return False
    if repetition_score(text) <= threshold:
        return False

    return True

def is_unique_hash(example):
    text = example["text"]
    if not isinstance(text, str):
        return False
    h = hashlib.md5(text.encode("utf-8")).hexdigest()
    if h in seen:
        return False
    seen.add(h)
    return True

def drop_bad_text(ds):
    # remove duplicates
    ds = ds.filter(is_unique_hash)
    # filter
    ds = ds.filter(is_good_example)
    return ds

wiki_ds = drop_bad_text(wiki_ds)
wiki_ds

Dataset({
    features: ['text'],
    num_rows: 1482386
})

In [ ]:
# import re 
# from datasets import Dataset

# CLEAN_RE = re.compile(r'[^a-zа-яё0-9 ]')

# def repetition_score(text):
#     tokens = CLEAN_RE.sub(' ', text.lower()).split()
#     return len(set(tokens)) / len(tokens) if tokens else 0.0

# def drop_bad_text(df):
#     # ----  drop deduplicates ----
#     df = df.drop_duplicates().copy()

#     min_cont_len = 200
#     # ----  drop short text ----
#     df = df[df["text"].str.len() > min_cont_len].copy()

#     # ---- drop text with formula like {...} ---- 
#     pattern = r"[{}]"
#     mask = df['text'].str.contains(pattern, regex=True, na=False)
#     df = df[~mask].copy()

#     # ---- drop text with list of source ----
#     pattern = r"//[^//]+//"
#     mask = df['text'].str.contains(pattern, regex=True, na=False)
#     df = df[~mask].copy()

#     # ---- drop repetition text ---- 
#     threshold = 0.6
#     df = df[df['text'].map(repetition_score) > threshold]
#     return df

# wiki_df = drop_bad_text(wiki_df)
# wiki_ds = Dataset.from_pandas(wiki_df[["text"]])
# wiki_df = ''
# wiki_ds

In [ ]:
# import pandas as pd

# splits = {'train': 'sberquad/train-00000-of-00001.parquet', 'validation': 'sberquad/validation-00000-of-00001.parquet', 'test': 'sberquad/test-00000-of-00001.parquet'}
# df = pd.read_parquet("hf://datasets/kuznetsoffandrey/sberquad/" + splits["test"])
# # df.to_json("./data/sberquad/sberquad_test.json", orient="records", force_ascii=False)

##### Download SberQuAD

In [3]:
import pandas as pd
df1 = pd.read_json('./raw_datasets/sberquad/sberquad_train.json')
df2 = pd.read_json('./raw_datasets/sberquad/sberquad_test.json')
df3 = pd.read_json('./raw_datasets/sberquad/sberquad_valid.json')
sberquad_df = pd.concat([df1, df2, df3], ignore_index=True)   
sberquad_df = sberquad_df[["context", "question"]].copy()
sberquad_df

,context,question
0,В протерозойских отложениях органические остат...,чем представлены органические остатки?
1,В протерозойских отложениях органические остат...,что найдено в кремнистых сланцах железорудной ...
2,В протерозойских отложениях органические остат...,что встречается в протерозойских отложениях?
3,В протерозойских отложениях органические остат...,что относится к числу древнейших растительных ...
4,В протерозойских отложениях органические остат...,как образовалось графито-углистое вещество?
...,...,...
74295,Классическая трёхуровневая система накачки раб...,Где используется классическая трёхуровневая си...
74296,Первая платёжная карта American Express появил...,какой инвестиционный банк входил в состав Amer...
74297,Следующий альбом Heroes был во многом созвучен...,С каким альбомом был созвучен альбом Дэвида Бо...
74298,"Одним из тех, на кого игра Данна произвела неи...",В каком техасском оркестре выступал гитарист Л...


In [4]:
from datasets import Dataset

sberquad_df["question_len"] = sberquad_df["question"].apply(lambda x: len(x))
# сортируем по длине вопроса (по убыванию)
sberquad_df = sberquad_df.sort_values(["context", "question_len"], ascending=False)
# удаляем дубликаты context, оставляя первый (самый длинный ответ)
sberquad_df = sberquad_df.drop_duplicates(subset="context", keep="first")
# Первая буква должны быть большой
sberquad_df["context"] = sberquad_df["context"].str[:1].str.upper() + sberquad_df["context"].str[1:]
# оставляем только нужные столбцы
sberquad_df["text"] = sberquad_df["question"] + " " + sberquad_df["context"]
# convert to Dataset
sberquad_ds = Dataset.from_pandas(
    sberquad_df[["text"]],
    preserve_index=False
)
# drop bad text
sberquad_ds = drop_bad_text(sberquad_ds)
sberquad_ds

Filter:   0%|          | 0/13489 [00:00<?, ? examples/s]

Filter:   0%|          | 0/13489 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 13154
})

##### Download WikiOmnia

In [6]:
from datasets import load_dataset, concatenate_datasets

wo1_ds = load_dataset(
    "json",
    data_files={
        "train": "https://huggingface.co/datasets/RussianNLP/wikiomnia/resolve/main/dummy/wikiomnia_ruT5_filtered/wikiomnia_ruT5_filtered_train.json"
    }
)["train"].select_columns(["summary"]).rename_column("summary", "text")

wo2_ds = load_dataset(
    "json",
    data_files={
        "train": "https://huggingface.co/datasets/RussianNLP/wikiomnia/resolve/main/dummy/wikiomnia_ruGPT3_filtered/wikiomnia_ruGPT_3_filtered_train.json"
    }
)["train"].select_columns(["summary"]).rename_column("summary", "text")

wo3_ds = load_dataset(
    "json",
    data_files={
        "train": "https://huggingface.co/datasets/RussianNLP/wikiomnia/resolve/main/dummy/wikiomnia_ruT5_raw/wikiomnia_dev.json"
    }
)["train"].select_columns(["summary"]).rename_column("summary", "text")

wo4_ds = load_dataset(
    "json",
    data_files={
        "train": "https://huggingface.co/datasets/RussianNLP/wikiomnia/resolve/main/dummy/wikiomnia_ruT5_raw/wikiomnia_test.json"
    }
)["train"].select_columns(["summary"]).rename_column("summary", "text")

wo_ds = concatenate_datasets([wo1_ds, wo2_ds, wo3_ds, wo4_ds])  
wo1_ds = wo2_ds = wo3_ds = wo4_ds = None
# drop bad text
wo_ds = drop_bad_text(wo_ds)
wo_ds

Filter:   0%|          | 0/2795387 [00:00<?, ? examples/s]

Filter:   0%|          | 0/582879 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 410448
})

#### Load lenta-ru-news dataset

In [11]:
from datasets import load_dataset

lenta_ds = load_dataset(
    "csv",
    data_files="./raw_datasets/lenta-ru-news.csv",
    split="train"
).select_columns(["text"])

# drop bad text
lenta_ds = drop_bad_text(lenta_ds)
lenta_ds

Filter:   0%|          | 0/800975 [00:00<?, ? examples/s]

Filter:   0%|          | 0/800037 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 793136
})

#### Load wikisource-creative-ru dataset

In [12]:
from datasets import load_dataset
wikisource_ds = load_dataset("kristaller486/wikisource-creative-ru")
wikisource_ds = wikisource_ds['train']
wikisource_ds = wikisource_ds.map(
    lambda example: {"text": example["original_text"]},  # создаём поле text
    remove_columns=wikisource_ds.column_names  
)
wikisource_ds = drop_bad_text(wikisource_ds)
wikisource_ds

Filter:   0%|          | 0/48427 [00:00<?, ? examples/s]

Filter:   0%|          | 0/48423 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 15284
})

#### QA датасет "HC3-ru"

In [ ]:
import re
from datasets import load_dataset

qa_ds = load_dataset("d0rj/HC3-ru", split="train")
pattern = r"(Пожалуйста,\s*)?(объясните|объясни),?\s*(как\s*)?будто\s*мне\s*пять\.?$"

def preprocess(example):
    # очистка вопроса
    question = re.sub(pattern, "", example["question"], flags=re.IGNORECASE).strip()
    
    # если chatgpt_answers пуст, берем human_answers
    answers = example["chatgpt_answers"] or example["human_answers"]
    
    if not answers:   # если нет ответов вообще
        return {"text": None}

    # выбираем самый длинный ответ
    answer = max(answers, key=len)

    return {"text": f"{question} {answer}"}

# применяем и удаляем все колонки кроме text
qa_ds = qa_ds.map(preprocess, remove_columns=qa_ds.column_names)
qa_ds = qa_ds.filter(lambda x: x["text"] is not None)
qa_ds

Map:   0%|          | 0/24322 [00:00<?, ? examples/s]

Filter:   0%|          | 0/24322 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 24322
})

#### Convert and concatenate datasets

In [13]:
from datasets import concatenate_datasets
    
combined_ds = concatenate_datasets([
    wiki_ds,
    sberquad_ds,
    wo_ds,
    lenta_ds,
    wikisource_ds,
])

combined_ds

Dataset({
    features: ['text'],
    num_rows: 2714408
})

#### Clean dataset

In [23]:
from pretrain.lm_cleaner import DataCleaner

cleaner = DataCleaner(
    min_length=300,
    min_cyrilic_ratio=0.8
)

clean_ds = cleaner.process_dataset(combined_ds)   
clean_ds

Map:   0%|          | 0/2714408 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2714408 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 1238501
})

In [24]:
chars = set("".join(clean_ds["text"]))
print(len(chars))

symbols = sorted([chars])
print(symbols)  

161
[{'—', 'A', 'i', 'И', 'X', '^', '№', 'E', 'v', 'М', '\n', 'f', 'J', ')', 'б', 'ю', 'l', '>', 'Ч', 'Ь', 'N', 'u', 'L', 'h', '.', 'b', 'З', 'j', '9', 'Z', 'c', '*', 'В', 'Ё', 'г', 'О', '2', "'", '=', 'ы', '/', 'т', 'm', ' ', 'ч', '5', 'у', 'a', 'С', 'W', 'о', '$', '(', 'd', '_', 'M', 'ж', 'з', 'р', '7', '8', 'n', 'х', 'Ы', '–', '…', 'Г', 'e', ':', 'o', 'в', 'Р', 'п', '#', '0', 'T', 'y', '\\', 'Я', 'П', 's', 'ъ', 'У', 'Б', 'P', 'Ю', 'я', '"', 'Д', 'S', 'Ц', 'Щ', 'ь', 'C', 'Ъ', 'p', '-', 'k', '6', 'м', 'и', '[', 'е', '?', 'ф', 'Й', '+', 'л', '%', 'G', 'Х', '3', 'r', '4', 'Ш', 'I', 'Л', 'R', 'а', 'Н', 'U', 'Т', 'V', ';', 'Е', 'н', '<', 'ц', 'O', 'g', 'q', 't', '@', 'щ', ']', '1', 'Ж', 'z', '!', 'x', 'д', '&', 'Q', 'B', 'с', 'ё', 'Э', 'ш', 'к', 'К', 'А', 'й', 'K', 'D', 'Y', ',', 'э', 'H', 'F', 'w', 'Ф'}]


#### SAP dataset

In [27]:
from docx import Document
import re


class DocxParser:

    def __init__(self, path: str):
        self.path = path
        self.chunks: list[str] = []

    # ---------- helpers ----------

    def _normalize(self, text: str) -> str:
        text = text.replace("\xa0", " ")
        text = re.sub(r"[ \t]+", " ", text)
        return text.strip()

    def _is_list_item(self, paragraph) -> bool:
        p = paragraph._p
        return p.pPr is not None and p.pPr.numPr is not None

    # ---------- main ----------

    def parse(self):
        doc = Document(self.path)
        current_chunk = []

        for paragraph in doc.paragraphs:
            raw_text = paragraph.text
            text = self._normalize(raw_text)

            if not text:
                continue

            # detect heading 1
            if paragraph.style.name.startswith("Heading 1"):
                if current_chunk:
                    self.chunks.append(
                        # "<BOS>\n" +
                        "\n".join(current_chunk) # +
                        # "\n<EOS>"
                    )
                    current_chunk = []

                current_chunk.append(text + "\n")  # separating the header
                continue

            # restore list marker
            if self._is_list_item(paragraph):
                text = "- " + text

            current_chunk.append(text)

        # last chunk
        if current_chunk:
            self.chunks.append(
                # "<BOS>\n" +
                "\n".join(current_chunk) # +
                # "\n<EOS>"
            )

        return self.chunks

#### Load SAP dataset

In [ ]:
parser = DocxParser("./raw_datasets/sap_generated/SAPHelp_CO.docx")
samples = parser.parse()

print(len(samples))
print(samples[228])

231
Контрольный список для создания мест возникновения затрат

Вы создали контроллинговую единицу и присвоили ей одну или несколько балансовых единиц. Контроллинговой единице присвоен верхний узел стандартной иерархии (см. Ведение контроллинговых единиц в пользовательской настройке Общего контроллинга).
Если требуется использовать пользовательские виды мест возникновения затрат, необходимо определить их в пользовательской настройке Учета по местам возникновения затрат (CO-OM-CCA) (см. Ведение видов мест возникновения затрат).
В зависимости от видов МВЗ введены значения по умолчанию для планирования и проводки данных первичных и вторичных затрат или облиго.
Проверьте эти значения по умолчанию:
- Определена стандартная иерархия (см. Ведение стандартной иерархии в пользовательской настройке Учета по местам возникновения затрат).
- При необходимости проверьте существующие средства поиска (см. Определение средств поиска для мест возникновения затрат в пользовательской настройке Учета по мес

In [30]:
sap_ds = Dataset.from_dict({"text": samples})
sap_ds

Dataset({
    features: ['text'],
    num_rows: 231
})

#### Load SAP dataset from QA samples

In [35]:
import json
from datasets import Dataset

# load json
with open("finetune/qa_samples_val.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# collect all docs
texts = []
for item in data:
    texts.extend(item.get("positive_docs", []))
    texts.extend(item.get("negative_docs", []))

# deduplicate  
texts = list(dict.fromkeys(texts))

# convert to Dataset
sap2_ds = Dataset.from_dict({"text": texts})
sap2_ds

Dataset({
    features: ['text'],
    num_rows: 864
})

#### Concatenate all datasets

In [40]:
from datasets import concatenate_datasets
    
full_ds = concatenate_datasets([
    clean_ds,
    sap_ds,
    sap2_ds
])

full_ds

Dataset({
    features: ['text'],
    num_rows: 1239596
})

#### Split dataset to train, valid, test

In [48]:
train_test = full_ds.train_test_split(test_size=0.003, seed=42)
train_valid = train_test["train"].train_test_split(test_size=0.003, seed=42)

dataset = {
    "train": train_valid["train"],
    "valid": train_valid["test"],
    "test": train_test["test"]
}
dataset

{'train': Dataset({
     features: ['text'],
     num_rows: 1232169
 }),
 'valid': Dataset({
     features: ['text'],
     num_rows: 3708
 }),
 'test': Dataset({
     features: ['text'],
     num_rows: 3719
 })}

#### Save dataset to json

In [49]:
dataset['train'].to_json("pretrain/combined_corpus_train.json")
dataset['valid'].to_json("pretrain/combined_corpus_valid.json")
dataset['test'].to_json("pretrain/combined_corpus_test.json")

Creating json from Arrow format:   0%|          | 0/1233 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

27686225

#### Training tokenizer

In [56]:
from pathlib import Path
import sys

proj_root = Path("..").resolve()     
if str(proj_root) not in sys.path:
    sys.path.append(str(proj_root))

from tokenizer import BPETokenizer

tokenizer = BPETokenizer(save_dir='../tokenizer/bpe_tokenizer_v0', vocab_size=32000, min_frequency=2)
bpe_tokenizer = tokenizer.train(full_ds)

Training BPE: 100%|██████████| 1239596/1239596 [23:44<00:00, 870.08it/s] 


Tokenizer saved!


Download tokenizer from file

In [57]:
bpe_tokenizer = BPETokenizer().from_file("../tokenizer/bpe_tokenizer_v0/tokenizer.json")

print(bpe_tokenizer.get_vocab_size())
encoded = bpe_tokenizer.encode("Привет мир")
print(encoded.ids)
print(encoded.tokens)

32000
[1, 28374, 316, 2700, 2]
['<BOS>', 'ÐŁÑĢÐ¸Ð²', 'ÐµÑĤ', 'ĠÐ¼Ð¸ÑĢ', '<EOS>']


In [58]:
txt = 'Задача на системы счисления, код выдаёт ошибку.'
bpe_tokenizer.encode(txt).ids

[1, 25934, 10898, 322, 2587, 928, 390, 801, 24, 4669, 7091, 1347, 21705, 26, 2]

In [59]:
dataset = load_dataset("json",
    data_files={
        "train": "pretrain/combined_corpus_train.json",
        "valid": "pretrain/combined_corpus_valid.json",
        # "test": "pretrain/combined_corpus_test.json"
    }
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating valid split: 0 examples [00:00, ? examples/s]

In [60]:
import numpy as np
from tqdm import tqdm

def build_bin(tokenizer, dataset, output_path, batch_size=10000, dtype=np.uint16):
    with open(output_path, "wb") as f:
        for i in tqdm(range(0, len(dataset), batch_size)):
            batch = dataset[i:i+batch_size]['text']  
            encoded_batch = tokenizer.encode_batch(batch)

            all_ids = []
            for row in encoded_batch:
                all_ids.extend(row.ids)

            np.array(all_ids, dtype=dtype).tofile(f)

build_bin(bpe_tokenizer, dataset['train'], 'pretrain/embed_corpus_train')
build_bin(bpe_tokenizer, dataset['valid'], 'pretrain/embed_corpus_valid')

100%|██████████| 1/1 [00:02<00:00,  2.40s/it]


In [3]:
import numpy as np
from pretrain.lm_dataset import LMDataset
gen_dataset = LMDataset(
    path='pretrain/embed_corpus_test',
    block_size=1024,
    dtype=np.uint16
)
print(len(gen_dataset))
print(gen_dataset[0]["input_ids"].shape)

17898
torch.Size([1024])


In [67]:
from torch.utils.data import DataLoader

gen_loader = DataLoader(
    gen_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

In [68]:
batch = next(iter(gen_loader))

print(batch["input_ids"].shape)
batch["input_ids"][0] 

torch.Size([2, 1024])


tensor([16289,  1198, 29455,  ...,    14,  4004,   289])

In [ ]:
# import re
# from datasets import Dataset

# class DataCleaner_:
#     def __init__(self, min_length=500):
#         self.min_length = min_length
#         self.citation_pattern = re.compile(r"\[\d+\]")
#         self.parentheses_pattern = re.compile(r"\([^()]*\)")
#         self.space_before_punct = re.compile(r"\s+([,.;:])")
#         self.double_symbols = re.compile(r"([.,/:;!~$@%^&*#+=\-]){2,}")

    
#     def clean_text(self, text: str) -> str:
#         if not text:
#             return ""

#         # Remove citation markers like [1], [23]
#         text = self.citation_pattern.sub("", text)

#         # Remove nested parentheses iteratively
#         while self.parentheses_pattern.search(text):
#             text = self.parentheses_pattern.sub("", text)

#         # replace multiple symbols with one
#         text = self.double_symbols.sub(r"\1", text)

#         # Remove space before punctuation: " ," -> ","
#         text = self.space_before_punct.sub(r"\1", text)
         
#         text = re.sub(r'\xad', '', text)

#         # Normalize whitespace
#         text = " ".join(text.split())
#         return text
    

#     def is_valid(self, text: str) -> bool:
#         if len(text) < self.min_length:
#             return False
        
#         # Optional: filter pages with too many non-letter symbols
#         letters_ratio = sum(c.isalpha() for c in text) / max(len(text), 1)
#         if letters_ratio < 0.6:
#             return False

#         return True
    

#     def process_dataset(self, dataset: Dataset) -> Dataset:
#         def process(example):
#             text = self.clean_text(example["text"])
#             if self.is_valid(text):
#                 return {"text": text}
#             else:
#                 return {"text": None}

#         dataset = dataset.map(process)
#         dataset = dataset.filter(lambda x: x["text"] is not None)
#         return dataset
